# 01b · Extract Celeb-DF face crops (cross-dataset test set)

Celeb-DF is **never used for training**. It's an unseen test set: models trained on FaceForensics++ are tested on it
to measure how well they generalise to a different deepfake generator, different people and different video quality.
Only the **official test split** (`List_of_testing_videos.txt`, 518 videos) is used, so results are comparable with papers.
The same face detector and crop as FF++ are used.

**Notebook settings (right sidebar)**
- **Accelerator:** None (CPU is enough)
- **Internet:** On
- **Inputs:** a **Celeb-DF (v2)** video dataset (`Celeb-real/`, `YouTube-real/`, `Celeb-synthesis/`, `List_of_testing_videos.txt`)

**Produces:** `/kaggle/working/data` (Celeb-DF test face crops + `manifest.csv`). Use **Save Version → Save & Run All (Commit)**.

In [ ]:
# --- Project code -------------------------------------------------------------------------
# Always clones the latest code. It lives in /tmp, OUTSIDE /kaggle/working, so it is never saved into this
# notebook's output (otherwise later notebooks would pick up a stale copy). For a private repo add a
# GITHUB_TOKEN secret. Fallback: a zip of the project uploaded as a Kaggle *dataset*.
import os, sys, glob, shutil, subprocess

PROJECT = "/tmp/dfdc-project"
REPO, BRANCH = "https://github.com/Amay-M-Nair/DFDC.git", "dev"

if not os.path.exists(PROJECT):
    token = ""
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        pass
    url = REPO.replace("https://", f"https://{token}@") if token else REPO
    r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, PROJECT], capture_output=True)
    if r.returncode == 0:
        subprocess.run(["git", "-C", PROJECT, "remote", "set-url", "origin", REPO])  # don't keep the token
    else:
        found = [p for d in range(5) for p in glob.glob("/kaggle/input/" + "*/" * d + "src/config.py")]
        if not found:
            raise RuntimeError("git clone failed: add a GITHUB_TOKEN secret or attach the project zip as a dataset")
        print("WARNING: git clone failed; using the attached copy at", os.path.dirname(os.path.dirname(found[0])))
        shutil.copytree(os.path.dirname(os.path.dirname(found[0])), PROJECT,
                        ignore=shutil.ignore_patterns("data", "outputs", "__pycache__"))

shutil.rmtree("/kaggle/working/project", ignore_errors=True)  # remove stale copies saved by older notebook versions
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
!pip install -q grad-cam anthropic
!git -C {PROJECT} log -1 --format="code version: %h  %s  (%cd)" --date=short 2>/dev/null || echo "code version: attached copy (no git)"

In [ ]:
from src.kaggle_utils import setup
setup(data_dir="/kaggle/working/data")

import json, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Markdown, Image as Show

## Settings
`FRAMES` frames are taken from each video (more frames → more stable video-level scores).

In [ ]:
FRAMES = 32
USE_ALL_VIDEOS = False   # True = every Celeb-DF video instead of the official 518-video test list

import glob
hits = glob.glob("/kaggle/input/**/Celeb-synthesis", recursive=True)
assert hits, "No Celeb-synthesis folder found: add a Celeb-DF v2 dataset as an input"
CDF_SRC = os.path.commonpath([os.path.dirname(h) for h in hits])
print("Celeb-DF root:", CDF_SRC)
print("test list found:", bool(glob.glob(CDF_SRC + "/**/List_of_testing_videos.txt", recursive=True)))

In [ ]:
args = f'--src "{CDF_SRC}" --out /kaggle/working/data --frames {FRAMES} --workers 4'
if USE_ALL_VIDEOS: args += " --all"
!python -m src.classifier.celebdf_extract {args}

## What was extracted

In [ ]:
m = pd.read_csv("/kaggle/working/data/manifest.csv")
display(m.groupby("label").agg(frames=("file", "count"), videos=("video", "nunique")))
fig, axes = plt.subplots(2, 8, figsize=(17, 4.6))
for row, label in zip(axes, ["real", "fake"]):
    for ax, r in zip(row, m[m.label == label].sample(8, random_state=0).itertuples()):
        ax.imshow(Image.open(f"/kaggle/working/data/test/{r.label}/{r.file}")); ax.axis("off")
        ax.set_title(f"{label}\n{r.video}", fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
import shutil
shutil.rmtree("/kaggle/working/outputs", ignore_errors=True)   # keep only the data in this notebook's output
!du -sh /kaggle/working/data